<a href="https://colab.research.google.com/github/Raka7317/set_project_work/blob/main/phising_Url%20detection(12feb).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ======================================
# 1. Mount Google Drive
# ======================================
from google.colab import drive
drive.mount("/content/drive")

# ======================================
# 2. Imports
# ======================================
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ======================================
# 3. Correct file paths (VERY IMPORTANT)
# ======================================
BASE_PATH = "/content/drive/MyDrive/data"

TRAIN_PATH = os.path.join(BASE_PATH, "train.csv")
TEST_PATH = os.path.join(BASE_PATH, "test.csv")
PHIUSIIL_PATH = os.path.join(BASE_PATH, "phiusiil.csv")  # Removed leading space

# ======================================
# 4. Safety check (prevents FileNotFoundError)
# ======================================
for path in [TRAIN_PATH, TEST_PATH, PHIUSIIL_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ File not found: {path}")

print("✅ All dataset files found")

# ======================================
# 5. Load & clean dataset
# ======================================
def load_and_clean(path):
    df = pd.read_csv(path)
    print(f"Columns in {os.path.basename(path)}: {df.columns.tolist()}") # Debugging line
    if os.path.basename(path) == "phiusiil.csv": # Changed check to match corrected filename
        df = df[['URL', 'label']].rename(columns={'URL': 'url', 'label': 'labels'}) # Rename for consistency
    else:
        df = df[['url', 'labels']]          # keep only required columns
    df.dropna(inplace=True)
    return df

train_df = load_and_clean(TRAIN_PATH)
test_df = load_and_clean(TEST_PATH)
phiusiil_df = load_and_clean(PHIUSIIL_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("PhiUSIIL shape:", phiusiil_df.shape)

# ======================================
# 6. Vectorization (URL → numerical)
# ======================================
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2
)

X_train = vectorizer.fit_transform(train_df['url'])
y_train = train_df['labels'] # corrected 'label' to 'labels'

X_test = vectorizer.transform(test_df['url'])
y_test = test_df['labels'] # corrected 'label' to 'labels'

X_phi = vectorizer.transform(phiusiil_df['url'])
y_phi = phiusiil_df['labels'] # corrected 'label' to 'labels'

# ======================================
# 7. Online Model (FTRL-style using SGD)
# ======================================
model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    learning_rate="optimal",
    random_state=42
)

print("🚀 Online training started...")
model.fit(X_train, y_train)
print("✅ Training completed")

# ======================================
# 8. Evaluation function
# ======================================
def evaluate(X, y, name):
    y_pred = model.predict(X)
    print(f"\n📊 Results on {name}")
    print("Accuracy :", accuracy_score(y, y_pred))
    print("Precision:", precision_score(y, y_pred))
    print("Recall   :", recall_score(y, y_pred))
    print("F1-score :", f1_score(y, y_pred))

# ======================================
# 9. Evaluate on all datasets
# ======================================
evaluate(X_test, y_test, "Test Set")
evaluate(X_phi, y_phi, "PhiUSIIL External Dataset")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All dataset files found
Columns in train.csv: ['labels', 'url']


/tmp/ipython-input-3701534857.py:38: DtypeWarning: Columns (16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Columns in test.csv: ['labels', 'url', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34']
Columns in phiusiil.csv: ['URL', 'label']
Train shape: (160000, 2)
Test shape: (160000, 2)
PhiUSIIL shape: (235795, 2)
🚀 Online training started...
✅ Training completed

📊 Results on Test Set
Accuracy : 0.93293125
Precision: 0.9108725309923483
Recall   : 0.959775
F1-score : 0.9346845613074044

📊 Results on PhiUSIIL External Dataset
Accuracy : 0.767225768146059
Precision: 0.7107228502911956
Recall   : 0.9999925843529848
F1-score : 0.83090099665727
